## Часть 1. Хэширование

### Задача: множество с операциями add / remove / contains

Есть пустое множество целых чисел. Нужно обрабатывать запросы:

* `+ x` - добавить `x`
* `- x` - удалить `x` (если есть)
* `? x` - проверить наличие: вывести `YES` или `NO`

Цель: сделать быстро даже при большом числе запросов.

### Решение с помощью небольшого массива (когда ключи из маленького диапазона)

Если известно, что ключи лежат в диапазоне `0..M` (например, `M = 10^6`), то можно хранить булев массив:

* `present[x] = True/False`

Тогда:

* `add`, `remove`, `contains` работают за `O(1)` (очень быстро),
* но память `O(M)` → если ключи “произвольные” (до `10^9`, строки, большие числа), так нельзя.

In [ ]:
+ 0
- 1
? 2

In [ ]:
data = input().split()
len(data)

+ sell - buy ? record


6

In [ ]:
data = input().split()
n = len(data)
M = 10**6 # максимальный ключ
present = [False] * (M + 1)

idx = 0
out = []
for idx in range(0, n, 2):
  op = data[idx]
  x = data[idx + 1]
  if 0 < x < M + 1:
    if op == "+":
      present[x] = True
    elif op == "-":
      present[x] = False
    elif op == "?":
      out.append("YES" if present[x] else "NO")

print(out.join("\n"))


### Обобщение: произвольные ключи → хэш-функция

Идея хэш-таблицы:

1. есть массив `buckets` длины `m` (например, 2^k)
2. каждый ключ `x` превращаем в индекс `h(x) mod m`
3. кладём ключ в соответствующий “бакет” (корзину)

Так мы “раскидываем” огромный мир ключей по компактному массиву.


### Определение хэш-функции

**Хэш-функция** `h(x)` - отображение ключей (чисел/строк/объектов) в целые числа так, чтобы ключи распределялись **как можно равномернее**.

Обычно нам нужен индекс:
$
idx = h(x) \bmod m
$

Свойства, которые хочется получить от хэш-функции:

* быстро считается,
* равномерно распределяет,
* небольшие изменения ключа сильно меняют хэш.



### Проблема коллизий

**Коллизия** - ситуация, когда разные ключи попадают в один и тот же бакет:

$$
x \ne y,\quad h(x)\bmod m = h(y)\bmod m
$$

Коллизии неизбежны (бакетов конечное число), поэтому хэш-таблица - это всегда:

* **хэш + механизм разрешения коллизий**.




### Как решают коллизии в хэш-таблицах

Два основных подхода:

1. **Открытая адресация** (open addressing):
   все элементы лежат прямо в массиве, а при коллизии ищем “следующее” место (делать это можно разными способами, свои изыскания можете начать с linear probing / quadratic probing / double hashing).
   *Очень коротко:* данные хранятся в одном массиве - это хорошо для кэша процессора, но при заполнении таблицы поиск новых мест замедляется.

2. **Метод цепочек** (chaining):
   в каждом бакете хранится список элементов (цепочка).
   *Этот метод разберём подробно*.




## Метод цепочек (chaining)

### Описание

* Есть массив `buckets` длины `m`
* `buckets[i]` - список (цепочка) ключей, которые попали в этот индекс
* Операции:

  * `add(x)` → идём в бакет и добавляем, если ещё нет
  * `contains(x)` → ищем в бакете
  * `remove(x)` → удаляем из бакета, если есть

### Почему это работает быстро

Если хэш-функция равномерная, то средняя длина цепочки примерно:
$$
\alpha = \frac{n}{m}
$$
где `n` - число элементов, `m` - число бакетов (load factor).


### Асимптотики chaining

Пусть `α = n/m`.

* В среднем:

  * `add / remove / contains` работают за **O(1 + α)**
  * при поддержании `α ≈ const` (например, ≤ 1–2) получаем **ожидаемо O(1)**

* В худшем случае (если всё упало в один бакет): **O(n)**
  (поэтому важен подбор хэш-функция и иногда выполнение rehash/resize)

### Реализация chaining: поиск / вставка / удаление


Ниже - простая реализация хеш-таблицы с цепочками для целых чисел.

Идея такая:

* берём массив из `m` бакетов,
* индекс бакета считаем как `x % m`,
* в каждом бакете храним список элементов.

Если два разных числа дают одинаковый остаток по модулю `m`, они попадают в один и тот же бакет.
Это и есть **коллизия**. Метод цепочек разрешает коллизии тем, что хранит все такие элементы в одном списке.

Для операций:

* `contains(x)` - идём в бакет `x % m` и проверяем, лежит ли там `x`,
* `add(x)` - если `x` ещё нет в нужном бакете, добавляем,
* `remove(x)` - если `x` есть в бакете, удаляем.



In [ ]:
import sys

class ChainingHashSet:
    def __init__(self, m=101):
        self.m = m
        self.buckets = [[] for _ in range(m)]

    def _idx(self, x):
        return hash(x) % self.m

    def contains(self, x):
        bucket = self.buckets[self._idx(x)]
        return x in bucket

    def add(self, x):
        bucket = self.buckets[self._idx(x)]
        if x not in bucket:
          bucket.append(x)

    def remove(self, x):
        bucket = self.buckets[self._idx(x)]
        if x in bucket:
          bucket.remove(x)


### Резюме по хешированию

* Массив работает очень быстро, но только когда ключи лежат в маленьком диапазоне.
* Для произвольных ключей используют хеш-таблицу: каждому ключу сопоставляется индекс бакета `idx = h(x) mod m`..
* Коллизии неизбежны: разные ключи могут попасть в один и тот же бакет. Значит нужен механизм разрешения коллизий
* Метод цепочек хранит все элементы одного бакета в списке.
* При равномерном распределении ключей операции `add`, `remove`, `contains` работают ожидаемо за `O(1)`.




## Часть 2. Алгоритм Рабина-Карпа

Теперь та же задача поиска подстроки, но через **хэши строк**.



### Постановка: поиск подстроки в строке

Даны строки `T` (текст) и `P` (шаблон).
Нужно найти все позиции, где `P` встречается в `T` (включая перекрытия).

Ограничения: `|T|, |P|` до `2e5` (или больше), поэтому нужен алгоритм около `O(n)`.



### Приближённое решение: сравнение хэшей

Идея:

* посчитать хэш`P`
* посчитать хэшкаждого подотрезка `T[i..i+m-1]`
* если хэши равны - считаем, что подстроки равны

Плюс: быстро.
Минус: возможны **коллизии** (разные строки → одинаковый хэш).

На практике:

* берут большой модуль (или два модуля),
* и/или при совпадении хэша делают финальную проверку `T[i:i+m] == P`.



## Полиномиальный хэшстрок

Пусть строка кодируется числами `c[i]` (например, `ord(s[i])`).

Определение (rolling hash):
$
H(s) = (c_0\cdot b^{n-1} + c_1\cdot b^{n-2} + \dots + c_{n-1}) \bmod M
$
или эквивалентно “слева направо”:
$
pref[i+1] = (pref[i]\cdot b + c_i) \bmod M
$



### Как считать быстро

1. **хэш всей строки за O(n)** - считаем массив `pref` и степени `pow_b`.

2. **хэш подстроки за O(1)**:
   Для полуинтервала `[l, r)`:
   $
   hash(l,r) = pref[r] - pref[l]\cdot b^{(r-l)} \pmod M
   $

Это и есть ключ к Рабину-Карпу.


In [ ]:
def build_hash(s: str, base: int, mod: int):
    n = len(s)
    pref = [0] * (n + 1)
    pows = [1] * (n + 1)

    for i, ch in enumerate(s):
      pref[i + 1] = (pref[i] * base + ord(ch)) % mod
      pows[i + 1] = (pows[i] * base) % mod

    return pref, pows


def sub_hash(pref, pows, l: int, r: int, mod: int):
    return (pref[r] - pref[l] * pows[r - l]) % mod

## Алгоритм Рабина-Карпа

### Идея

* посчитать хэш`P`
* посчитать префикс-хэши `T`
* для каждого `i` сравнить хэш`T[i..i+m)` с хэшем `P`
* при совпадении - (опционально) проверить подстроку напрямую

Чтобы резко уменьшить вероятность коллизий, часто берут **два модуля**.


In [ ]:
import sys

def build_hash(s: str, base: int, mod: int):
    n = len(s)
    pref = [0] * (n + 1)
    pows = [1] * (n + 1)

    for i, ch in enumerate(s):
      pref[i + 1] = (pref[i] * base + ord(ch)) % mod
      pows[i + 1] = (pows[i] * base) % mod

    return pref, pows

def sub_hash(pref, pows, l: int, r: int, mod: int):
    return (pref[r] - pref[l] * pows[r - l]) % mod

def rabin_karp_all(text: str, pattern: str):
    n, m = len(text), len(pattern)
    if m == 0:
      return list(range(n + 1))
    if m > n:
      return []

    base = 911382323
    mod = 1000000001
    prefs_t, pows_t = build_hash(text, base, mod)
    prefs_p, pows_p = build_hash(pattern, base, mod)
    hp = prefs_p[m]

    res = []
    for i in range(n - m + 1):
      ht = sub_hash(prefs_t, pows_t, i, i + m, mod)
      if hp != ht:
        continue

      if pattern == text[i: i + m]:
        res.append(i)

    return res

data = sys.stdin.read().splitlines()
T = 'abaccccaba'
P = 'aba'
pos = rabin_karp_all(T, P)
print(len(pos))
if pos:
    print(*pos)





2
0 7


### Асимптотика Рабина-Карпа

Пусть `n = |T|`, `m = |P|`.

* Построение префикс-хэшей: `O(n)`
* хэшшаблона: `O(m)`
* Проверка всех позиций: `O(n)` сравнений хэша за `O(1)` каждое
* Итог: **O(n + m)** ожидаемо

Финальная проверка `text[i:i+m] == pattern` вызывается редко (только при совпадении хэшей).



## Вероятность коллизий и выбор параметров

Если хэшравномерный, то вероятность случайной коллизии для одного модуля примерно `~ 1 / M`.

* При `M ≈ 10^9` это уже очень мало.
* При двух модулях вероятность порядка:
  $
  \approx \frac{1}{M_1 M_2} \sim 10^{-18}
  $
  что практически безопасно.

Практические советы:

* берите большой простой модуль (или два),
* base выбирайте не маленький (и не кратный модулю),
* для избежания коллизий делайте финальную проверку строк при совпадении хэша.


### Резюме по Рабину-Карпу

* Идея: сравниваем хэши вместо строк.
* Префикс-хэши дают хэш любой подстроки за `O(1)`.
* Алгоритм: `O(n+m)` ожидаемо.
* Коллизии минимизируются двумя модулями и/или финальной проверкой.


## Задача: изоморфизм строк

Даны две строки `s` и `t` одинаковой длины.

Нужно проверить, можно ли заменить каждый символ строки `s` на некоторый символ так, чтобы получилась строка `t`, причём:

- одинаковые символы в `s` должны переходить в одинаковые символы в `t`,
- разные символы в `s` должны переходить в разные символы в `t`.

### Идея решения

Будем хранить два словаря:

- `to_t[c]` — в какой символ строки `t` должен переходить символ `c` из `s`,
- `to_s[c]` — обратное соответствие.

Идём по строкам слева направо.  
Для каждой пары символов `(a, b)`:

- если `a` уже сопоставлялся с другим символом, ответ `NO`,
- если `b` уже сопоставлялся с другим символом, ответ `NO`,
- иначе запоминаем соответствие.

Если противоречий не возникло, ответ `YES`.

### Асимптотика

- Время: `O(n)`
- Память: `O(k)`, где `k` — число различных символов

In [ ]:
def is_isomorphic(s: str, t: str) -> bool:
    if len(s) != len(t):
      return False

    to_s = {}
    to_t = {}

    for a,b in zip(s, t):
      if a in to_s and to_s[a] != b:
        return False
      if b in to_t and to_t[b] != a:
        return False

      to_s[a], to_t[b] = b, a

    return True

## Задача: почти одинаковые строки

Даны `n` строк одинаковой длины.

Нужно определить, существуют ли две различные строки, которые отличаются ровно в одной позиции.

### Идея решения

Если две строки отличаются ровно в одной позиции `i`, то после удаления символа в позиции `i` они становятся одинаковыми.

Значит, для каждой строки и для каждой позиции `i` хочется быстро получить хеш строки, из которой удалили символ `s[i]`.

Если делать это напрямую как `s[:i] + s[i+1:]`, то одно такое действие стоит `O(m)`, и всё решение получится слишком медленным.

Поэтому используем **полиномиальные хеши строк**.

Для каждой строки заранее считаем:

- массив префикс-хешей,
- массив степеней основания.

Тогда хеш любой подстроки можно получать за `O(1)`.

Хеш строки без символа `s[i]` состоит из двух частей:

- префикс `s[0:i]`,
- суффикс `s[i+1:m]`.

Их можно склеить в один общий хеш за `O(1)`:

`hash_without_i = hash(left) * base^(len(right)) + hash(right)`

Для каждой строки и каждой позиции `i` будем строить ключ:

- номер позиции `i`,
- хеш строки без символа в этой позиции.

Если такой ключ уже встречался раньше у строки с **другим** символом в позиции `i`, значит найдены две строки, отличающиеся ровно в одной позиции.

### Почему это корректно

Если две строки отличаются ровно в одной позиции `i`, то после удаления символа в этой позиции они совпадут, а удалённые символы будут различными.

И наоборот, если для одной и той же позиции `i` у двух строк совпала строка после удаления `i`-го символа, но сами удалённые символы различны, то эти строки отличаются ровно в одной позиции.

### Асимптотика

Пусть:

- `n` — число строк,
- `m` — длина каждой строки.

Тогда:

- построение хешей для одной строки: `O(m)`,
- обработка всех удалений для одной строки: `O(m)`,
- итоговое время: `O(n * m)`,
- память: `O(n * m)` в худшем случае.

In [ ]:
def build_hash(s: str, base: int, mod: int):
    n = len(s)
    pref = [0] * (n + 1)
    pows = [1] * (n + 1)

    for i, ch in enumerate(s):
      pref[i + 1] = (pref[i] * base + ord(ch)) % mod
      pows[i + 1] = (pows[i] * base) % mod

    return pref, pows

def sub_hash(pref, pows, l: int, r: int, mod: int):
    return (pref[r] - pref[l] * pows[r - l]) % mod


def exists_pair_with_one_difference(arr):
    if not arr:
      return False

    m = len(arr[0])

    base = 911382323
    mod = 1_000_000_007
    seen = {}

    for s in arr:
      prefs, pows = build_hash(s, base, mod)

      for i in range(m):
        left = sub_hash(prefs, pows, 0, i, mod)
        right = sub_hash(prefs, pows, i+1, m, mod)
        len_right = m - i - 1
        merged = left * pows[len_right] + right

        key = (i, merged)
        ch = s[i]

        if key in seen:
          if ch not in seen[key]:
            return True
          seen[key].add(ch)
        else:
          seen[key] = {ch}

    return False



Главная мысль здесь такая:

* наивно удаление символа даёт `O(m)` на одну позицию,
* позиций `m`, строк `n`,
* получаем `O(n * m^2)`.

С хешами:

* каждое “удаление” обрабатывается через два хеша подстрок и одну склейку,
* то есть за `O(1)`,
* итог `O(n * m)`.


## Задача: самый длинный зеркальный фрагмент

Назовём подстроку зеркальной, если она читается одинаково слева направо и справа налево.

Дана строка `s`. Нужно найти длину её самой длинной зеркальной подстроки.

### Идея решения

У каждого палиндрома есть центр:

- либо один символ — для палиндромов нечётной длины,
- либо промежуток между двумя соседними символами — для палиндромов чётной длины.

Можно для каждого центра пытаться расширяться влево и вправо, пока символы совпадают.  
Такой подход понятный, но работает за `O(n^2)`.

Чтобы решить задачу за `O(n)`, используют **алгоритм Манакера**.

Он заранее хранит:

- `d1[i]` — радиус максимального нечётного палиндрома с центром в `i`,
- `d2[i]` — радиус максимального чётного палиндрома с центром между `i-1` и `i`.

После этого длина ответа легко восстанавливается:

- нечётный палиндром: `2 * d1[i] - 1`
- чётный палиндром: `2 * d2[i]`

### Асимптотика

- Время: `O(n)`
- Память: `O(n)`

In [ ]:
def manacher(s: str):
    pass


def longest_palindrome_length(s: str) -> int:
    d1, d2 = manacher(s)

    # TODO